# **Clean Dataset**

## **Import Data**

In [ ]:
import pandas as pd
import numpy as np
from PIL import Image
from wordcloud import WordCloud, STOPWORDS, ImageColorGenerator
import matplotlib.pyplot as plt
from collections import Counter
import seaborn as sns
import matplotlib.pyplot as plt
import os
import re
import json
import string
import nltk
nltk.download('punkt')
nltk.download('stopwords')
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
%pip install Sastrawi
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from nltk.stem import PorterStemmer
from nltk.stem.snowball import SnowballStemmer

In [ ]:
# Load file CSV
url = 'https://drive.google.com/uc?id=1AquDcY68zFoltVYPjd6p2CLeVOYZRB6U'
tka = pd.read_csv(url)
tka.head()

In [ ]:
tka.info()

In [ ]:
# Kolom 'full_text'
df = tka[['full_text']]
df.head()

In [ ]:
# Menghapus duplikasi data
df.drop_duplicates(subset='full_text', keep = 'first', inplace=True)
df.info()

# **Preprocessing Data**

## _**Casefolding**_

In [ ]:
def case_folding(text):
    if isinstance(text, str):
      lowercased_text = text.lower()
      return lowercased_text
    else:
      return text

df['casefolding'] = df['full_text'].apply(case_folding)
df.head(5)

## _**Cleaning data**_

In [ ]:
# Menghapus Emoji dan Bendera
def remove_emoji(text):
    if text is not None and isinstance(text, str):
        emoji_pattern = re.compile("["
            u"\U0001F600-\U0001F64F"  # emoticons
            u"\U0001F300-\U0001F5FF"  # symbols & pictographs
            u"\U0001F680-\U0001F6FF"  # transport & map symbols
            u"\U0001F700-\U0001F77F"  # alchemical symbols
            u"\U0001F780-\U0001F7FF"  # Geometric Shapes Extended
            u"\U0001F800-\U0001F8FF"  # Supplemental Arrows-C
            u"\U0001F900-\U0001F9FF"  # Supplemental Symbols and Pictographs
            u"\U0001FA70-\U0001FAFF"  # Symbols and Pictographs Extended-A
            u"\U00002702-\U000027B0"  # Dingbats
            u"\U000024C2-\U0001F251"  # Enclosed characters
            u"\U0001F1E0-\U0001F1FF"  # Regional indicator symbol (bendera)
            "]+", flags=re.UNICODE)
        return emoji_pattern.sub(r'', text)
    return text
# Menghapus Emotikon berbasis karakter seperti :) :( :v
def remove_emoticon(text):
    if text is not None and isinstance(text, str):
        emoticon_pattern = r'[:;=8xX][-^o*]?[)(DPpOo3|/\\]'
        return re.sub(emoticon_pattern, '', text)
    return text
# Menghapus Username (mention)
def remove_username(text):
    return re.sub(r'@\w+', '', text)
# Menghapus URL atau Link
def remove_links(text):
    return re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
# Menghapus Simbol selain huruf dan angka
def remove_symbols(text):
    if text is not None and isinstance(text, str):
        return re.sub(r'[^a-zA-Z0-9\s]', '', text)
    return text
# Menghapus Angka
def remove_numbers(text):
    return re.sub(r'\d+', '', text)
# Menghapus kata berulang (misal: "aaaaaa" jadi "a", "haaaallo" jadi "halo")
def remove_repeated_chars(text):
    return re.sub(r'(.)\1{2,}', r'\1', text)
# Mengubah newline atau enter (\n) menjadi spasi
def remove_newline(text):
    return text.replace('\n', ' ').replace('\r', ' ')
# Menghapus spasi berlebihan
def remove_extra_whitespace(text):
    return re.sub(r'\s+', ' ', text).strip()
# Menghapus trigger word menfess seperti sch!, ptn!, stud!, dll.
def remove_menfess(text):
    if text is not None and isinstance(text, str):
        trigger_words = ["sch", "ptn", "stud", "colle","buten"]
        pattern = r'\b(' + '|'.join(trigger_words) + r')\s*!+'
        text = re.sub(pattern, '', text, flags=re.IGNORECASE)
        return text
    return text
# Gabungkan semua fungsi jadi satu pipeline
def clean_text(text):
    text = remove_username(text)
    text = remove_links(text)
    text = remove_emoji(text)
    text = remove_emoticon(text)
    text = remove_menfess(text)
    text = remove_symbols(text)
    text = remove_numbers(text)
    text = remove_repeated_chars(text)
    text = remove_newline(text)
    text = remove_extra_whitespace(text)
    return text
df['cleaning'] = df['casefolding'].apply(clean_text)
# Hapus baris yang kosong setelah dibersihkan
df = df[df['cleaning'].str.strip() != '']
df.head()

## _**Normalization**_ **(Mengubah kata tidak baku menjadi baku)**

In [ ]:
def replace_taboo_words(text, kamus_tidak_baku):
    if isinstance(text, str):
        words = text.split()
        replaced_words = []
        kata_diganti = []
        kata_baku = []

        for word in words:
            word_clean = word.lower().strip()

            if word_clean in kamus_tidak_baku:
                baku_word = kamus_tidak_baku[word_clean]
                replaced_words.append(baku_word)
                kata_diganti.append(word_clean)
                kata_baku.append(baku_word)
            else:
                replaced_words.append(word)

        replaced_text = ' '.join(replaced_words)
        return replaced_text, kata_baku, kata_diganti

    return '', [], []

In [ ]:
# Menggunakan kamuskatabaku dari www.kaggle.com/datasets/fornigulo/kamuskatabaku
sheet_id = "1z68haLNKapFm18s0o_76XKdJs-ykd3gO"
kamuskatabaku = f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=xlsx"
kamus_data = pd.read_excel(kamuskatabaku)
kamus_tidak_baku = dict(zip(kamus_data['tidak_baku'], kamus_data['kata_baku']))

In [ ]:
df['normalized'], df['kata_baku'], df['kata_tidak_baku'] = zip(
    *df['cleaning'].apply(lambda x: replace_taboo_words(x, kamus_tidak_baku))
)

df = df[['full_text', 'casefolding', 'cleaning', 'normalized']]
df.head(5)

In [ ]:
# Menampilkan baris indeks 0, 3, dan 4
df.iloc[[0, 3, 4]]

## **Drop data kosong dan duplikasi**

In [ ]:
data = df.dropna()
data.info()

In [ ]:
# Hitung jumlah baris sebelum menghapus duplikasi
before_drop = len(df)
# Hapus duplikasi berdasarkan kolom 'normalized'
df.drop_duplicates(subset='normalized', keep='first', inplace=True)
# Hitung jumlah baris setelah menghapus duplikasi
after_drop = len(df)
# Hitung jumlah baris yang dihapus
dropped = before_drop - after_drop
print(f"Jumlah baris yang dihapus: {dropped}")

In [ ]:
df.info()

In [ ]:
# Ambil kolom yang dibutuhkan
df_ready = df[['full_text', 'normalized']]
df_ready.head()

In [ ]:
# Simpan ke file CSV
df_ready.to_csv("tka_cleaned.csv", index=False)

# **Labeling**

In [ ]:
# Library umum
import pandas as pd
import torch
from tqdm import tqdm

# Transformers
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.nn.functional import softmax

In [ ]:
# Load data hasil preprocessing
df = pd.read_csv("tka_cleaned.csv")
df.head()

In [ ]:
# Gunakan model BERT pretrained
model_name = "mdhugol/indonesia-bert-sentiment-classification"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)
model.eval();  # masuk ke mode evaluasi

In [ ]:
# Lihat mapping label
print(model.config.id2label)

# Mapping label
label_map = {
    0: 'positive',
    1: 'neutral',
    2: 'negative'
}

In [ ]:
# Confidence threshold
THRESHOLD = 0.7

def predict_sentiment(text):
    clean_text = str(text).lower().strip()

    # Jika hanya "tka" atau "tes kompetensi akademik", auto netral
    if clean_text in ["tka", "tes kompetensi akademik", "tes kemampuan akademik"]:
        return 1, 1.0

    # Prediksi dengan model
    inputs = tokenizer(
        clean_text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=64
    )

    with torch.no_grad():
        outputs = model(**inputs)
        probs = softmax(outputs.logits, dim=1)
        max_prob, predicted = torch.max(probs, dim=1)

        predicted = predicted.item()
        max_prob = max_prob.item()
        return predicted, label_map[predicted], max_prob

In [ ]:
# Simpan jumlah data awal sebelum filtering
jumlah_awal = len(df)

# Tampilkan progress bar
tqdm.pandas()

# Prediksi sentimen
results = df['normalized'].progress_apply(predict_sentiment)

# Pisahkan hasil prediksi
df['label_num_all'] = results.apply(lambda x: x[0])
df['pseudo_label_all'] = results.apply(lambda x: x[1])
df['confidence_score'] = results.apply(lambda x: x[2])

# Distribusi sebelum threshold
print("Jumlah data sebelum threshold:", len(df))
print("\nDistribusi label sebelum threshold:")
print(df['pseudo_label_all'].value_counts())

print("\nDistribusi label sebelum threshold (%):")
print((df['pseudo_label_all'].value_counts(normalize=True) * 100).round(2))

In [ ]:
# Seleksi berdasarkan threshold
labeled_df = df[df['confidence_score'] >= THRESHOLD].copy()
low_confidence_df = df[df['confidence_score'] < THRESHOLD].copy()

# Setelah threshold, gunakan nama kolom final
labeled_df['label_num'] = labeled_df['label_num_all'].astype(int)
labeled_df['pseudo_label'] = labeled_df['pseudo_label_all']

# Reset index
labeled_df.reset_index(drop=True, inplace=True)
low_confidence_df.reset_index(drop=True, inplace=True)

In [ ]:
# Statistik filtering
print("\nJumlah data awal:", jumlah_awal)
print("Jumlah data lolos threshold:", len(labeled_df))
print("Jumlah data dibuang:", len(low_confidence_df))
print("Persentase data dibuang:", round(len(low_confidence_df) / jumlah_awal * 100, 2), "%")


In [ ]:
# Distribusi setelah threshold
print("\nDistribusi label setelah threshold:")
print(labeled_df['pseudo_label'].value_counts())

print("\nDistribusi label setelah threshold (%):")
print((labeled_df['pseudo_label'].value_counts(normalize=True) * 100).round(2))


In [ ]:
# Distribusi data yang dibuang berdasarkan prediksi awal
print("\nDistribusi data confidence rendah berdasarkan prediksi awal:")
print(low_confidence_df['pseudo_label_all'].value_counts())

print("\nDistribusi data confidence rendah berdasarkan prediksi awal (%):")
print((low_confidence_df['pseudo_label_all'].value_counts(normalize=True) * 100).round(2))


In [ ]:
# Simpan hasil
df.to_csv("label_utuh.csv", index=False)
labeled_df.to_csv("tka_labeled.csv", index=False)
low_confidence_df.to_csv("low_conf.csv", index=False)


In [ ]:
# Tampilkan beberapa data confidence rendah
display(low_confidence_df[['normalized', 'pseudo_label_all', 'confidence_score']].head(20))

In [ ]:
# Hasil label
labeled_df[['normalized', 'label_num', 'pseudo_label','confidence_score']].head()

# Label final

Data yang tidak lolos treshold dilabeli secara manual dan digabungkan kembali ke data awal


In [ ]:
import pandas as pd
label_final = pd.read_csv('final_labeled.csv')

In [ ]:
# Distribusi data yang dibuang berdasarkan prediksi awal
print("\nDistribusi data confidence rendah berdasarkan prediksi awal:")
print(label_final['pseudo_label'].value_counts())

print("\nDistribusi data confidence rendah berdasarkan prediksi awal (%):")
print((label_final['pseudo_label'].value_counts(normalize=True) * 100).round(2))


# Pembagian data

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Memuat dataset
tweet_labeled = pd.read_csv('final_labeled.csv')
# Memisahkan data latih (70%) dan sisanya (30%)
train_data, temp_data = train_test_split(tweet_labeled, test_size=0.3, random_state=42,
                                         stratify=tweet_labeled['label_num'])

# Memisahkan data validasi (10%) dan data uji (20%) dari sisa data
val_data, test_data = train_test_split(temp_data, test_size=2/3, random_state=42,
                                       stratify=temp_data['label_num'])

# Menyimpan masing-masing dataset ke file CSV
train_data.to_csv('train_dataset.csv', index=False)
val_data.to_csv('val_dataset.csv', index=False)
test_data.to_csv('test_dataset.csv', index=False)

# Menampilkan jumlah data masing-masing
print(f"Jumlah data latih: {len(train_data)}")
print(f"Jumlah data validasi: {len(val_data)}")
print(f"Jumlah data uji: {len(test_data)}")

## Wordcloud

In [ ]:
!pip install wordcloud

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from wordcloud import WordCloud
from collections import Counter

In [ ]:
label_map = {
    "positive": "positif",
    "neutral": "netral",
    "negative": "negatif",
    "positif": "positif",
    "netral": "netral",
    "negatif": "negatif"
}
df["sentimen"] = df["pseudo_label"].map(label_map)
print(df["sentimen"].value_counts())


In [ ]:
stopwords_id = set([
    "yang", "dan", "di", "ke", "dari", "ini", "itu", "untuk", "dengan",
    "karena", "pada", "ada", "jadi", "atau", "dalam", "akan", "bisa",
    "sudah", "belum", "saja", "aja", "aku", "saya", "gue", "gua", "gw",
    "kamu", "dia", "mereka", "kita", "kami", "nya", "nih", "dong",
    "sih", "deh", "lah", "pun", "ya", "yaitu", "adalah", "sebagai",
    "juga", "lebih", "kurang", "sangat", "banget", "bgt", "gak", "ga",
    "nggak", "tidak", "tak", "kalau", "kalo", "kan", "kok", "mah",
    "rt", "amp", "guys","sama", "mau", "buat", "apa", "semua", "orang",
    "banget", "lagi", "udah", "udahh", "aja", "cuma", "kayak","tapi",
    "kaya", "gitu", "gini", "dong", "nih","apa","bagaimana","kenapa",
    "kalian","masih","punya","harus","snbt","mata","kah","kan","kak",
    "tuh","banyak","tau","sampai","soalnya","habis","hari","benar"
])

# Kata khusus topik yang boleh dihapus jika terlalu dominan
stopwords_topik = set([
    "tka", "tes", "kemampuan", "akademik", "soal", "sekolah", "siswa",
    "ujian", "belajar","nilai","utbk","kelas","tiga","try","out"
])
stopwords_final = stopwords_id.union(stopwords_topik)

In [ ]:
def get_words(text_series, stopwords):
    all_text = " ".join(text_series.dropna().astype(str))
    words = all_text.split()

    cleaned_words = []
    for word in words:
        word = word.strip().lower()
        if len(word) > 2 and word not in stopwords:
            cleaned_words.append(word)

    return cleaned_words


def get_top_words(text_series, stopwords, top_n=20):
    words = get_words(text_series, stopwords)
    counter = Counter(words)
    return counter.most_common(top_n)

In [ ]:
from collections import Counter
import pandas as pd

def get_words(text_series, stopwords):
    all_text = " ".join(text_series.dropna().astype(str))
    words = all_text.split()

    cleaned_words = []
    for word in words:
        word = word.strip().lower()
        if len(word) > 2 and word not in stopwords:
            cleaned_words.append(word)

    return cleaned_words


def get_top_words(text_series, stopwords, top_n=20):
    words = get_words(text_series, stopwords)
    counter = Counter(words)
    return counter.most_common(top_n)


sentimen_order = ["negatif", "netral", "positif"]

top_words_all = []

for sentimen in sentimen_order:
    data_sentimen = df[df["sentimen"] == sentimen]
    top_words = get_top_words(data_sentimen["normalized"], stopwords_final, top_n=20)

    for rank, (word, freq) in enumerate(top_words, start=1):
        top_words_all.append({
            "sentimen": sentimen,
            "peringkat": rank,
            "kata": word,
            "frekuensi": freq
        })

top_words_df = pd.DataFrame(top_words_all)

top_10_words = []

for sentimen in sentimen_order:
    subset = top_words_df[top_words_df["sentimen"] == sentimen].head(30)
    kata_dominan = ", ".join(subset["kata"].tolist())

    top_10_words.append({
        "Sentimen": sentimen.capitalize(),
        "Kata Dominan": kata_dominan
    })

top_10_words_df = pd.DataFrame(top_10_words)
top_10_words_df

In [ ]:
def create_wordcloud(text_series, stopwords, title, filename):
    words = get_words(text_series, stopwords)
    text = " ".join(words)

    wordcloud = WordCloud(
        width=1200,
        height=800,
        background_color="white",
        stopwords=stopwords,
        collocations=False,
        max_words=100
    ).generate(text)

    plt.figure(figsize=(10, 6))
    plt.imshow(wordcloud, interpolation="bilinear")
    plt.axis("off")
    plt.title(title, fontsize=16)
    plt.tight_layout()
    plt.savefig(filename, dpi=300, bbox_inches="tight")
    plt.show()

In [ ]:
for sentimen in sentimen_order:
    data_sentimen = df[df["sentimen"] == sentimen]

    create_wordcloud(
        text_series=data_sentimen["normalized"],
        stopwords=stopwords_final,
        title=f"Word Cloud Sentimen {sentimen.capitalize()}",
        filename=f"wordcloud_sentimen_{sentimen}.png"
    )

# Gold standard


In [ ]:
import pandas as pd
from sklearn.metrics import cohen_kappa_score, confusion_matrix, classification_report

# Coba baca CSV dengan encoding yang lebih fleksibel
test_df = pd.read_csv("test_dataset.csv", encoding="latin1")


print(test_df.columns)

In [ ]:
import pandas as pd
from sklearn.metrics import cohen_kappa_score, confusion_matrix

# Pastikan kolom tidak ada spasi tersembunyi
test_df.columns = test_df.columns.str.strip()

# Ubah ke integer
test_clean["num_old"] = test_clean["num_old"].astype(int)
test_clean["label_num"] = test_clean["label_num"].astype(int)

# Hitung jumlah sesuai dan berbeda
jumlah_sesuai = (test_clean["num_old"] == test_clean["label_num"]).sum()
jumlah_berbeda = (test_clean["num_old"] != test_clean["label_num"]).sum()
total_data = len(test_clean)

persen_sesuai = jumlah_sesuai / total_data * 100
persen_berbeda = jumlah_berbeda / total_data * 100

print("Total data:", total_data)
print("Jumlah sesuai:", jumlah_sesuai)
print("Jumlah berbeda:", jumlah_berbeda)
print("Persentase sesuai:", persen_sesuai)
print("Persentase berbeda:", persen_berbeda)

# Cohen's Kappa
kappa = cohen_kappa_score(test_clean["label_num"], test_clean["num_old"])
print("Cohen's Kappa:", kappa)


In [ ]:
kesesuaian_df = pd.DataFrame({
    "Keterangan": [
        "Pseudo-label sesuai dengan gold standard",
        "Pseudo-label berbeda dengan gold standard",
        "Total"
    ],
    "Jumlah Data": [
        jumlah_sesuai,
        jumlah_berbeda,
        total_data
    ],
    "Persentase": [
        f"{persen_sesuai:.2f}%",
        f"{persen_berbeda:.2f}%",
        "100%"
    ]
})

kesesuaian_df

In [ ]:
label_names = ["positif", "netral", "negatif"]

crosstab = pd.crosstab(
    test_df["label_num"].map({0: "positif", 1: "netral", 2: "negatif"}),
    test_df["num_old"].map({0: "positif", 1: "netral", 2: "negatif"}),
    rownames=["Gold Standard"],
    colnames=["Pseudo-label"]
)

crosstab